# Paper-to-Code 2 – word2vec (2013)
**Papers:** Mikolov et al. – *Efficient Estimation of Word Representations in Vector Space* – [arXiv:1301.3781](https://arxiv.org/abs/1301.3781) and *Distributed Representations of Words and Phrases and their Compositionality* (negative sampling, subsampling) – [arXiv:1310.4546](https://arxiv.org/abs/1310.4546) · notes: [[Text Representations]] · plan: [[Paper-to-Code Months]]

### The claim we reproduce
> A shallow **skip-gram** model trained only to predict nearby words learns vectors where **similar words are close** and **relations are directions**: `vec(paris) − vec(france) + vec(germany) ≈ vec(berlin)`. Negative sampling + subsampling of frequent words make it cheap enough to train on huge corpora.

### Our scaled-down setup
**text8** (first 100 MB of cleaned English Wikipedia, 31 MB zip) – we use the first **4M tokens**, 100-d vectors, window 3, 5 negatives, **1 epoch** (≈ 2 min on a CPU). The paper trained on 6B tokens with 300-d vectors, so expect neighbours to work well and analogies to work *sometimes*.

**How this works:** cells marked **TODO** are yours (each one is a small function). Run the ✅ check under it.
Experiment cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*.
Everything runs on a laptop CPU – the paper's full-scale setup is listed under *Scale it up*.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, collections, pathlib, urllib.request, zipfile, time
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); torch.set_num_threads(4); rng = np.random.default_rng(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the experiment cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

path = pathlib.Path("data/text8.zip"); path.parent.mkdir(exist_ok=True)
if not path.exists():
    urllib.request.urlretrieve("http://mattmahoney.net/dc/text8.zip", path)
N_TOKENS = 4_000_000
words = zipfile.ZipFile(path).read("text8").decode().split()[:N_TOKENS]
counter = collections.Counter(words)
vocab = [w for w, c in counter.most_common() if c >= 5]            # min_count = 5, as in the paper's tool
w2i = {w: i for i, w in enumerate(vocab)}
counts = np.array([counter[w] for w in vocab])
ids = np.array([w2i[w] for w in words if w in w2i])
print(f"{len(words):,} tokens, vocabulary {len(vocab):,}, most common: {vocab[:8]}")

## 1. Subsampling frequent words
"the", "of", "and" appear millions of times and carry little information. The paper (Sec. 2.3 of 1310.4546) **discards** each occurrence of word $w$ with probability $1-\sqrt{t / f(w)}$, where $f(w)$ is its relative frequency and $t = 10^{-5}$ (we use $10^{-4}$ for a small corpus). Return the **keep** probability $\min(1, \sqrt{t/f(w)})$ for every vocabulary word.

In [ ]:
def keep_probability(counts, t=1e-4):
    # TODO 1: relative frequency f = counts / total, keep prob = min(1, sqrt(t / f))
    raise NotImplementedError  # TODO 1

In [ ]:
check("subsampling", lambda: np.allclose(keep_probability(np.array([9900, 99, 1]), t=1e-4), [np.sqrt(1e-4 / 0.99), np.sqrt(1e-4 / 0.0099), 1]),
      "For counts [9900, 99, 1] (f = 0.99, 0.0099, 0.0001) expect [0.01005, 0.1005, 1].")

## 2. (center, context) training pairs
Skip-gram predicts every word within `window` positions of the center word, in both directions. Return two equal-length arrays `centers, contexts` with **all** pairs (order doesn't matter). Vectorise it: for each offset `d = 1..window`, `ids[:-d]` and `ids[d:]` are aligned neighbours.

In [ ]:
def make_pairs(ids, window):
    # TODO 2: all (center, context) pairs with 1 <= |offset| <= window
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_pairs():
    c, o = make_pairs(np.array([7, 8, 9]), 2)
    return sorted(zip(c.tolist(), o.tolist())) == [(7, 8), (7, 9), (8, 7), (8, 9), (9, 7), (9, 8)]
check("pairs", _t_pairs, "For ids [7, 8, 9] and window 2 expect all 6 ordered pairs of different positions.")

## 3. The noise distribution
Negative samples are drawn from the unigram distribution raised to the **3/4 power** – it boosts rare words compared to raw frequency. The paper found it "outperformed significantly the unigram and the uniform distributions".

In [ ]:
def noise_distribution(counts, power=0.75):
    # TODO 3: counts**power, normalised to sum to 1
    raise NotImplementedError  # TODO 3

In [ ]:
check("noise distribution", lambda: np.allclose(noise_distribution(np.array([16, 1])), np.array([8, 1]) / 9),
      "counts [16, 1] → 16^0.75 = 8 and 1 → probabilities [8/9, 1/9].")

## 4. The negative-sampling loss (Eq. 4 of 1310.4546)
For a center vector $v_c$, the true context's output vector $u_o$ and $K$ noise words $u_1..u_K$:
$$\mathcal{L} = -\log\sigma(u_o^\top v_c) - \sum_{k=1}^{K}\log\sigma(-u_k^\top v_c)$$
Shapes: `v_center (B, D)`, `u_context (B, D)`, `u_negative (B, K, D)`. Return the batch mean. Use `F.logsigmoid` (numerically stable).

In [ ]:
def sgns_loss(v_center, u_context, u_negative):
    # TODO 4: the negative-sampling loss above, averaged over the batch
    raise NotImplementedError  # TODO 4

In [ ]:
def _t_loss():
    B, K, D = 3, 5, 4
    zero = sgns_loss(torch.zeros(B, D), torch.zeros(B, D), torch.zeros(B, K, D))          # every logit 0 → (1+K)·log 2
    v = torch.ones(1, D); good = sgns_loss(v, v, -v.expand(1, K, D))
    return abs(zero.item() - (1 + K) * np.log(2)) < 1e-5 and good.item() < 0.2
check("SGNS loss", _t_loss, "All-zero vectors must give (1+K)·log 2 = 4.159; aligned context + opposite negatives must give a small loss.")

## 5. Train
Two embedding tables like the paper: **input** vectors $v$ (the ones we keep) and **output** vectors $u$. Adam instead of the paper's linearly decaying SGD – simpler and fine at this scale.

In [ ]:
DIM, WINDOW, NEG, BATCH = 100, 3, 5, 4096
if ready("subsampling", "pairs", "noise distribution", "SGNS loss"):
    keep = ids[rng.random(len(ids)) < keep_probability(counts)[ids]]
    centers, contexts = make_pairs(keep, WINDOW)
    perm = rng.permutation(len(centers)); centers, contexts = torch.tensor(centers[perm]), torch.tensor(contexts[perm])
    noise = torch.tensor(noise_distribution(counts), dtype=torch.float)
    print(f"kept {len(keep):,} of {len(ids):,} tokens after subsampling → {len(centers):,} training pairs")

    V = len(vocab)
    emb_in, emb_out = nn.Embedding(V, DIM), nn.Embedding(V, DIM)
    nn.init.uniform_(emb_in.weight, -0.5 / DIM, 0.5 / DIM); nn.init.zeros_(emb_out.weight)   # word2vec's init
    opt = torch.optim.Adam([*emb_in.parameters(), *emb_out.parameters()], lr=0.003)
    t0, curve = time.time(), []
    for b in range(0, len(centers), BATCH):
        c, o = centers[b:b + BATCH], contexts[b:b + BATCH]
        negs = torch.multinomial(noise, len(c) * NEG, replacement=True).view(len(c), NEG)
        loss = sgns_loss(emb_in(c), emb_out(o), emb_out(negs))
        opt.zero_grad(); loss.backward(); opt.step(); curve.append(loss.item())
    print(f"done in {time.time()-t0:.0f}s, final loss {np.mean(curve[-100:]):.3f}")
    plt.plot(np.convolve(curve, np.ones(50) / 50, "valid")); plt.xlabel("step"); plt.ylabel("SGNS loss"); plt.show()

## 6. Did it learn meaning?

In [ ]:
def neighbours(word, k=6):
    s = E @ E[w2i[word]]
    return [vocab[i] for i in s.topk(k + 1).indices[1:].tolist()]

def analogy(a, b, c):   # a : b  ::  c : ?
    s = E @ F.normalize(E[w2i[b]] - E[w2i[a]] + E[w2i[c]], dim=0)
    for w in (a, b, c): s[w2i[w]] = -1
    return vocab[s.argmax().item()]

QUESTIONS = [("man", "king", "woman", "queen"), ("france", "paris", "germany", "berlin"), ("france", "paris", "italy", "rome"),
             ("germany", "berlin", "england", "london"), ("big", "bigger", "small", "smaller"), ("good", "better", "bad", "worse"),
             ("one", "two", "three", "four"), ("january", "february", "march", "april"), ("boy", "girl", "brother", "sister"),
             ("france", "french", "germany", "german"), ("italy", "italian", "spain", "spanish"), ("walk", "walking", "swim", "swimming")]

if ready("subsampling", "pairs", "noise distribution", "SGNS loss"):
    E = F.normalize(emb_in.weight.detach(), dim=1)
    for w in ["three", "january", "computer", "france", "water"]:
        print(f"{w:>9} → {neighbours(w)}")
    n_ok = 0
    for a, b, c, d in QUESTIONS:
        p = analogy(a, b, c); n_ok += p == d
        print(f"{'✓' if p == d else '·'} {a}:{b} :: {c}:{p}  (want {d})")
    print(f"analogies: {n_ok}/{len(QUESTIONS)}  (random guessing ≈ 0/{len(QUESTIONS)} with a {len(vocab):,}-word vocabulary)")

In [ ]:
NUMBERS = {"one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "zero"}
MONTHS = {"january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"}
if ready("subsampling", "pairs", "noise distribution", "SGNS loss"):
    check("similar words are close", lambda: len(NUMBERS & set(neighbours("three"))) >= 3 and len(MONTHS & set(neighbours("january"))) >= 3,
          "At least 3 of the 6 neighbours of 'three' should be numbers and of 'january' months.")
    check("relations are directions", lambda: n_ok >= 3,
          "Expected at least 3/12 analogies right – check the loss sign and that you use the INPUT vectors.")

<details><summary>▶ Solution</summary>

```python
def keep_probability(counts, t=1e-4):
    f = counts / counts.sum()
    return np.minimum(1.0, np.sqrt(t / f))
```

```python
def make_pairs(ids, window):
    centers, contexts = [], []
    for d in range(1, window + 1):
        centers += [ids[:-d], ids[d:]]
        contexts += [ids[d:], ids[:-d]]
    return np.concatenate(centers), np.concatenate(contexts)
```

```python
def noise_distribution(counts, power=0.75):
    p = counts.astype(float) ** power
    return p / p.sum()
```

```python
def sgns_loss(v_center, u_context, u_negative):
    pos = (v_center * u_context).sum(-1)                              # (B,)
    neg = torch.bmm(u_negative, v_center.unsqueeze(2)).squeeze(2)     # (B, K)
    return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(1)).mean()
```
</details>

## Scale it up (stretch goals)
- All 17M tokens of text8, 3–5 epochs, window 5 with the paper's **dynamic window** (sample the window size uniformly per center word), 300-d vectors.
- Evaluate on the full Google analogy set (`questions-words.txt` from the original word2vec release, ~19.5k questions; the paper's Table 6 reports ~60 % for skip-gram on 783M words).
- Ablations: power 0.75 vs 1.0 vs 0 (uniform), with vs without subsampling, number of negatives 2/5/15.
- Use $v+u$ (input + output vectors) as the embedding – does it help?

---
## 📝 Write-up (fill in – half a page is enough)
Copy this into your results log in [[Paper-to-Code Months]] when done.

- **Paper and its claim in one sentence:**
- **What I implemented / what I scaled down (and why):**
- **Result:**

  | setting | metric | paper | me |
  |---|---|---|---|
  |  |  |  |  |

- **Did the claim hold at small scale? Why / why not?**
- **One detail the paper glossed over that I had to figure out:**
- **What I would try next with a GPU:**
